*0.4 Deep learning basics*

# Seq2seq + attention

**The situation.** The seq2seq date model fails most on the long formats: by the time the decoder writes the day, the day digits from the start of the input have been squeezed out of the one summary vector. Translation systems in 2014 had the same failure on long sentences — and the fix, from a 2015 paper, changed everything after it.

**Attention in a seq2seq.** Keep *every* encoder output, not just the last. At each decoding step, let the decoder score all of them against its current state (a dot product), softmax the scores into weights, and read a weighted sum. When writing the month, it looks at "March"; when writing the day, at "3". The bottleneck is gone, and the weights show where the model looked — this is the attention from 0.2, in its first home.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import random
from datetime import date, timedelta

import torch

random.seed(0)
MONTHS = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]


def random_pair():
    day = date(2000, 1, 1) + timedelta(days=random.randint(0, 365 * 30))
    formats = [
        f"{MONTHS[day.month - 1]} {day.day}, {day.year}",
        f"{day.day}/{day.month}/{day.year % 100:02d}",
        f"{day.day} {MONTHS[day.month - 1][:3]} {day.year}",
        f"{day.month:02d}-{day.day:02d}-{day.year}",
        f"{day.day}th of {MONTHS[day.month - 1]} {day.year}",
    ]
    return random.choice(formats), day.isoformat()


pairs = []
for _ in range(4000):
    pairs.append(random_pair())
all_text = ""
for source_text, target_text in pairs:
    all_text += source_text + target_text
alphabet = ["<pad>", "<start>", "<end>"] + sorted(set(all_text))
index = {}
for position, character in enumerate(alphabet):
    index[character] = position
PAD, START, END = 0, 1, 2


def encode(text, length):
    ids = []
    for character in text[:length]:
        ids.append(index[character])
    return ids + [PAD] * (length - len(ids))


IN_LEN, OUT_LEN = 24, 12
input_rows = []
target_rows = []
for source_text, target_text in pairs:
    input_rows.append(encode(source_text, IN_LEN))
    target_rows.append([START] + encode(target_text, OUT_LEN - 2) + [END])
inputs = torch.tensor(input_rows)
targets = torch.tensor(target_rows)
train_inputs, val_inputs = inputs[:3600], inputs[3600:]
train_targets, val_targets = targets[:3600], targets[3600:]
print("input", tuple(inputs.shape), "| target", tuple(targets.shape))

input (4000, 24) | target (4000, 12)


**The model.** Same encoder and decoder as before, plus one attention step per output character. The attention weights are kept so they can be printed.

In [3]:
import time

import torch.nn.functional as F
from torch import nn


class Seq2SeqAttention(nn.Module):
    def __init__(self, alphabet_size, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(alphabet_size, 32, padding_idx=PAD)
        self.encoder = nn.GRU(32, hidden, batch_first=True)
        self.decoder = nn.GRUCell(32 + hidden, hidden)  # input char + what attention read
        self.output = nn.Linear(hidden, alphabet_size)

    def step(self, token, state, encoder_outputs, source_mask):
        scores = torch.bmm(encoder_outputs, state.unsqueeze(2)).squeeze(
            2
        )  # decoder state · every encoder output
        scores = scores.masked_fill(~source_mask, -1e9)  # never attend to padding
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(
            1
        )  # weighted sum of encoder outputs
        state = self.decoder(torch.cat([self.embedding(token), context], dim=1), state)
        return self.output(state), state, weights

    def forward(self, source, target_in):
        encoder_outputs, memory = self.encoder(self.embedding(source))
        state = memory[0]
        mask = source != PAD
        logits = []
        for position in range(target_in.shape[1]):
            step_logits, state, _ = self.step(target_in[:, position], state, encoder_outputs, mask)
            logits.append(step_logits)
        return torch.stack(logits, dim=1)

    @torch.no_grad()
    def predict(self, source):
        encoder_outputs, memory = self.encoder(self.embedding(source))
        state = memory[0]
        mask = source != PAD
        token = torch.full((source.shape[0],), START)
        out, attention = [], []
        for _ in range(OUT_LEN - 1):
            step_logits, state, weights = self.step(token, state, encoder_outputs, mask)
            token = step_logits.argmax(dim=-1)
            out.append(token)
            attention.append(weights)
        return torch.stack(out, dim=1), torch.stack(attention, dim=1)


def exact_match(model, source, target):
    model.eval()
    predicted, _ = model.predict(source)
    return (predicted == target[:, 1:]).all(dim=1).float().mean().item()


def train(model, epochs=12, lr=3e-3, batch_size=64):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(train_inputs))
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            logits = model(train_inputs[batch], train_targets[batch][:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                train_targets[batch][:, 1:].reshape(-1),
                ignore_index=PAD,
            )
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if epoch % 3 == 0:
            print(
                
                    f"epoch {epoch:>2}  loss {loss.item():.3f}  exact-match on validation "
                    f"{exact_match(model, val_inputs, val_targets):.1%}"
                
            )
    return exact_match(model, val_inputs, val_targets)


torch.manual_seed(0)
started = time.perf_counter()
model = Seq2SeqAttention(len(alphabet))
attention_accuracy = train(model)
print(f"trained in {time.perf_counter() - started:.0f} s")
assert attention_accuracy > 0.8

epoch  3  loss 0.079  exact-match on validation 74.8%


epoch  6  loss 0.074  exact-match on validation 90.8%


epoch  9  loss 0.024  exact-match on validation 98.0%


epoch 12  loss 0.001  exact-match on validation 100.0%
trained in 11 s


**Where did it look?** For one validation date, print the input character each output character attended to most.

In [4]:
source = val_inputs[:1]
predicted, attention = model.predict(source)
source_text = ""
for i in source[0].tolist():
    if i != PAD:
        source_text += alphabet[i]
print("input: ", source_text)
line = "output:"
for position, token in enumerate(predicted[0].tolist()):
    if token == END:
        break
    looked_at = int(attention[0, position].argmax())
    line += (
        f"\n  {alphabet[token]!r} ← looked at input position {looked_at:>2} "
        f"({source_text[looked_at] if looked_at < len(source_text) else ' '!r})"
    )
print(line)

input:  09-05-2015
output:
  '2' ← looked at input position  8 ('1')
  '0' ← looked at input position  7 ('0')
  '1' ← looked at input position  8 ('1')
  '5' ← looked at input position  9 ('5')
  '-' ← looked at input position  8 ('1')
  '0' ← looked at input position  0 ('0')
  '9' ← looked at input position  1 ('9')
  '-' ← looked at input position  0 ('0')
  '0' ← looked at input position  3 ('0')
  '5' ← looked at input position  4 ('5')


**Reading the output.** Higher exact-match than the plain seq2seq, and the attention trace shows the decoder reading the year digits when it writes the year, the month name when it writes the month. The model learned an alignment nobody labelled.

```
"March 3, 2024"  encoder outputs:  M a r c h _ 3 , _ 2 0 2 4
                                   ▲ ▲ ▲             ▲ ▲ ▲ ▲
writing "03" (month)  ─ weights ───┘ │ │             │ │ │ │
writing "2024"        ─ weights ─────────────────────┘ ┘ ┘ ┘     a different look per output step
```

**The rule to remember.** Attention lets the decoder look back at everything instead of trusting one summary. It fixed long inputs, made models explainable by their weights — and once someone asked "what if we use *only* attention?", the transformer followed.

| Use it when | Don't when | Instead use |
|---|---|---|
| understanding attention with a picture you can print | new systems | transformer (next items) |

**Watch out**
- Mask padding before the softmax or the decoder learns to read blanks.
- Attention weights explain *where* the model looked, not *why*; do not oversell them as explanations.
- This is *cross*-attention (decoder over encoder). *Self*-attention (a sequence over itself) is the transformer's step.